# HSGP Residual Baseline

This notebook tests the first Hilbert-space Gaussian process residual model for the wellbore TVT task. The mean model keeps the slope-core Bayesian linear baseline, while the HSGP component models smooth residual structure over boundary/progress geometry. This is the bridge from Bayesian linear baselines into richer Gaussian process families.


## 1. Setup

In [1]:
"Configure imports, paths, JAX, NumPyro, and deterministic settings."

from pathlib import Path
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
import numpyro.handlers as handlers
from numpyro.contrib.hsgp.approximation import hsgp_squared_exponential
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoDelta, AutoNormal
from numpyro.optim import Adam

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.validation import group_well_folds
from src.io.loaders import discover_wells, load_horizontal_well
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_TAG = "bayesian_hsgp_residual_baseline"
RUN_DATE = "2026-06-22"
SEED = 42
N_SPLITS = 3
MAX_TARGET_ROWS_PER_WELL = 140
MAX_TRAIN_ROWS_PER_FOLD = 50000
SVI_STEPS = 850
LEARNING_RATE = 0.02
POSTERIOR_SAMPLES = 120
RESIDUAL_SAMPLE_PER_FOLD = 1800

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
numpyro.set_platform("cpu")
plt.style.use("seaborn-v0_8-whitegrid")
print({"jax": jax.__version__, "numpyro": numpyro.__version__, "devices": [str(device) for device in jax.devices()]})


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'jax': '0.10.2', 'numpyro': '0.21.0', 'devices': ['cpu:0']}


## 2. Slope-Core And HSGP Coordinate Frame

The slope-core features match the previous Bayesian notebooks. The HSGP coordinates are standardized boundary/progress variables so the domain half-width `ell` can be fixed conservatively while the kernel amplitude and length scale are learned.


In [2]:
"Define notebook-local slope-core feature construction helpers."


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_slope_core_features_for_well(well, directory, max_rows):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    known_mask = horizontal["TVT_input"].notna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}
    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    target_positions = deterministic_sample_positions(target_mask, max_rows)
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)

    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "TVT": float(horizontal["TVT"].iloc[pos]),
            "endpoint_hold_pred": last_known_tvt,
        }
        row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


slope_features = ["tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100", "z_slope_last_25", "z_slope_last_50", "z_slope_last_100"]
slope_projection_features = ["tvt_slope_last_10_projected_delta", "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta"]
feature_columns = ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features + ["last_known_tvt"]
hsgp_coordinate_sets = {
    "boundary_progress": ["distance_log1p", "eval_progress"],
    "boundary_progress_horizontal": ["distance_log1p", "eval_progress", "horizontal_log1p"],
}

print({"n_mean_features": len(feature_columns), "coordinate_sets": hsgp_coordinate_sets})


{'n_mean_features': 17, 'coordinate_sets': {'boundary_progress': ['distance_log1p', 'eval_progress'], 'boundary_progress_horizontal': ['distance_log1p', 'eval_progress', 'horizontal_log1p']}}


In [3]:
"Build sampled target-zone frame and HSGP coordinate columns."

start = time.time()
train_wells = discover_wells(TRAIN_DIR)
frames = [build_slope_core_features_for_well(well, TRAIN_DIR, MAX_TARGET_ROWS_PER_WELL) for well in train_wells]
feature_frame = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
build_seconds = time.time() - start

feature_frame["distance_log1p"] = np.log1p(feature_frame["distance_from_boundary_md"].clip(lower=0.0))
feature_frame["horizontal_log1p"] = np.log1p(feature_frame["horizontal_dist_from_boundary"].clip(lower=0.0))

print({"wells": len(train_wells), "sample_rows": len(feature_frame), "build_seconds": round(build_seconds, 2)})
feature_frame.head()


13:55:04 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


{'wells': 773, 'sample_rows': 108220, 'build_seconds': 46.81}


,wellname,row_idx,md_norm,distance_from_boundary_md,eval_progress,last_known_tvt,dZ_from_boundary,horizontal_dist_from_boundary,TVT,endpoint_hold_pred,...,tvt_slope_last_50,tvt_slope_last_50_projected_delta,tvt_slope_last_100,tvt_slope_last_100_projected_delta,z_slope_last_10,z_slope_last_25,z_slope_last_50,z_slope_last_100,distance_log1p,horizontal_log1p
0,000d7d20,1442,0.273261,1.0,0.000000,11747.37,0.01,1.000450,11747.38,11747.37,...,0.016327,0.016327,0.008485,0.008485,0.007778,0.005833,0.002653,0.004343,0.693147,0.693372
1,000d7d20,1469,0.278378,28.0,0.007040,11747.37,0.39,27.992356,11747.52,11747.37,...,0.016327,0.457143,0.008485,0.237576,0.007778,0.005833,0.002653,0.004343,3.367296,3.367032
2,000d7d20,1497,0.283684,56.0,0.014342,11747.37,1.02,55.983960,11747.46,11747.37,...,0.016327,0.914286,0.008485,0.475152,0.007778,0.005833,0.002653,0.004343,4.043051,4.042770
3,000d7d20,1524,0.288800,83.0,0.021382,11747.37,1.80,82.975967,11748.05,11747.37,...,0.016327,1.355102,0.008485,0.704242,0.007778,0.005833,0.002653,0.004343,4.430817,4.430531
4,000d7d20,1552,0.294106,111.0,0.028683,11747.37,2.68,110.962778,11748.66,11747.37,...,0.016327,1.812245,0.008485,0.941818,0.007778,0.005833,0.002653,0.004343,4.718499,4.718166


## 3. HSGP Model Definitions

The HSGP term is a zero-mean residual function added to the standardized signed-delta target. We compare a linear Student-t reference against small HSGP basis configurations before trying richer GP variants.


In [4]:
"Define Bayesian linear and HSGP residual models with fold preparation helpers."


def linear_mu(X):
    n_features = X.shape[1]
    intercept = numpyro.sample("intercept", dist.Normal(0.0, 1.5))
    linear_beta = numpyro.sample("linear_beta", dist.Normal(0.0, 2.0).expand([n_features]).to_event(1))
    return intercept + jnp.dot(X, linear_beta)


def student_t_linear_reference(X, X_gp, y=None, hsgp_m=None, hsgp_ell=None):
    mu = linear_mu(X)
    sigma = numpyro.sample("sigma", dist.HalfNormal(1.5))
    numpyro.deterministic("mu", mu)
    numpyro.deterministic("hsgp_residual", jnp.zeros(X.shape[0]))
    numpyro.sample("obs", dist.StudentT(3.0, mu, sigma), obs=y)


def student_t_hsgp_residual(X, X_gp, y=None, hsgp_m=None, hsgp_ell=None):
    linear_part = linear_mu(X)
    hsgp_alpha = numpyro.sample("hsgp_alpha", dist.HalfNormal(0.75))
    hsgp_length = numpyro.sample("hsgp_length", dist.LogNormal(0.0, 0.45))
    with handlers.scope(prefix="hsgp"):
        residual_gp = hsgp_squared_exponential(
            x=X_gp,
            alpha=hsgp_alpha,
            length=hsgp_length,
            ell=hsgp_ell,
            m=hsgp_m,
            non_centered=True,
        )
    mu = linear_part + residual_gp
    sigma = numpyro.sample("sigma", dist.HalfNormal(1.5))
    numpyro.deterministic("mu", mu)
    numpyro.deterministic("hsgp_residual", residual_gp)
    numpyro.deterministic("hsgp_alpha_value", hsgp_alpha)
    numpyro.deterministic("hsgp_length_value", hsgp_length)
    numpyro.sample("obs", dist.StudentT(3.0, mu, sigma), obs=y)


candidate_specs = [
    {"candidate": "linear_t3_autodelta_reference", "model": student_t_linear_reference, "guide": "autodelta", "coord_set": "none", "hsgp_m": [], "hsgp_ell": []},
    {"candidate": "hsgp_bp_m5_autodelta", "model": student_t_hsgp_residual, "guide": "autodelta", "coord_set": "boundary_progress", "hsgp_m": [5, 5], "hsgp_ell": [3.0, 3.0]},
    {"candidate": "hsgp_bp_m7_autodelta", "model": student_t_hsgp_residual, "guide": "autodelta", "coord_set": "boundary_progress", "hsgp_m": [7, 7], "hsgp_ell": [3.0, 3.0]},
    {"candidate": "hsgp_bph_m4_autodelta", "model": student_t_hsgp_residual, "guide": "autodelta", "coord_set": "boundary_progress_horizontal", "hsgp_m": [4, 4, 3], "hsgp_ell": [3.0, 3.0, 3.0]},
    {"candidate": "hsgp_bp_m5_autonormal", "model": student_t_hsgp_residual, "guide": "autonormal", "coord_set": "boundary_progress", "hsgp_m": [5, 5], "hsgp_ell": [3.0, 3.0]},
]


def make_guide(spec):
    if spec["guide"] == "autonormal":
        return AutoNormal(spec["model"])
    return AutoDelta(spec["model"])


def fit_preprocessor(train_frame, columns):
    if len(columns) == 0:
        return pd.Series(dtype=float), pd.Series(dtype=float), pd.Series(dtype=float)
    medians = train_frame[columns].median(numeric_only=True)
    filled = train_frame[columns].fillna(medians)
    means = filled.mean()
    stds = filled.std().replace(0.0, 1.0).fillna(1.0)
    return medians, means, stds


def apply_preprocessor(frame, columns, medians, means, stds):
    if len(columns) == 0:
        return np.zeros((len(frame), 1), dtype=float)
    filled = frame[columns].fillna(medians)
    return ((filled - means) / stds).to_numpy(float)


def prepare_fold_arrays(train_frame, valid_frame, coord_set):
    x_medians, x_means, x_stds = fit_preprocessor(train_frame, feature_columns)
    X_train = apply_preprocessor(train_frame, feature_columns, x_medians, x_means, x_stds)
    X_valid = apply_preprocessor(valid_frame, feature_columns, x_medians, x_means, x_stds)
    gp_columns = hsgp_coordinate_sets.get(coord_set, [])
    gp_medians, gp_means, gp_stds = fit_preprocessor(train_frame, gp_columns)
    Xgp_train = apply_preprocessor(train_frame, gp_columns, gp_medians, gp_means, gp_stds)
    Xgp_valid = apply_preprocessor(valid_frame, gp_columns, gp_medians, gp_means, gp_stds)
    y_train_raw = train_frame["delta_tvt_from_boundary"].to_numpy(float)
    y_mean = float(np.mean(y_train_raw))
    y_std = float(np.std(y_train_raw) + 1e-9)
    y_train = (y_train_raw - y_mean) / y_std
    return {"X_train": jnp.asarray(X_train), "X_valid": jnp.asarray(X_valid), "Xgp_train": jnp.asarray(Xgp_train), "Xgp_valid": jnp.asarray(Xgp_valid), "y_train": jnp.asarray(y_train), "y_mean": y_mean, "y_std": y_std}


def metric_row(y_true, pred_tvt):
    return {"rmse": float(np.sqrt(mean_squared_error(y_true, pred_tvt))), "mae": float(mean_absolute_error(y_true, pred_tvt)), "r2": float(r2_score(y_true, pred_tvt))}


def interval_metrics(y_true, draws):
    q10 = np.quantile(draws, 0.10, axis=0)
    q90 = np.quantile(draws, 0.90, axis=0)
    q025 = np.quantile(draws, 0.025, axis=0)
    q975 = np.quantile(draws, 0.975, axis=0)
    return {"coverage_80": float(((y_true >= q10) & (y_true <= q90)).mean()), "coverage_95": float(((y_true >= q025) & (y_true <= q975)).mean()), "width_80": float(np.mean(q90 - q10)), "width_95": float(np.mean(q975 - q025))}


pd.DataFrame([{k: spec[k] for k in ["candidate", "guide", "coord_set", "hsgp_m", "hsgp_ell"]} for spec in candidate_specs])


,candidate,guide,coord_set,hsgp_m,hsgp_ell
0,linear_t3_autodelta_reference,autodelta,none,[],[]
1,hsgp_bp_m5_autodelta,autodelta,boundary_progress,"[5, 5]","[3.0, 3.0]"
2,hsgp_bp_m7_autodelta,autodelta,boundary_progress,"[7, 7]","[3.0, 3.0]"
3,hsgp_bph_m4_autodelta,autodelta,boundary_progress_horizontal,"[4, 4, 3]","[3.0, 3.0, 3.0]"
4,hsgp_bp_m5_autonormal,autonormal,boundary_progress,"[5, 5]","[3.0, 3.0]"


## 4. Grouped HSGP CV

In [5]:
"Run grouped well-fold validation for linear reference and HSGP residual candidates."

fit_start = time.time()
fold_rows = []
residual_rows = []
posterior_rows = []
wells = sorted(feature_frame["wellname"].unique())
folds = group_well_folds(wells, n_splits=N_SPLITS)

for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_frame = feature_frame.loc[feature_frame["wellname"].isin(train_fold_wells)].copy()
    valid_frame = feature_frame.loc[feature_frame["wellname"].isin(valid_fold_wells)].copy()
    train_frame = deterministic_row_cap(train_frame, MAX_TRAIN_ROWS_PER_FOLD)
    y_true_tvt = valid_frame["TVT"].to_numpy(float)
    endpoint_pred = valid_frame["endpoint_hold_pred"].to_numpy(float)
    fold_rows.append({"fold": fold_idx, "candidate": "endpoint_hold", "guide": "baseline", "coord_set": "baseline", "n_basis": 0, "n_train_rows": 0, "n_valid_rows": len(valid_frame), "final_loss": np.nan, "coverage_80": np.nan, "coverage_95": np.nan, "width_80": np.nan, "width_95": np.nan, "hsgp_alpha": np.nan, "hsgp_length": np.nan, "hsgp_residual_sd": np.nan, **metric_row(y_true_tvt, endpoint_pred)})

    for spec_idx, spec in enumerate(candidate_specs):
        arrays = prepare_fold_arrays(train_frame, valid_frame, spec["coord_set"])
        rng_key = jax.random.PRNGKey(SEED + fold_idx * 100 + spec_idx)
        guide = make_guide(spec)
        svi = SVI(spec["model"], guide, Adam(LEARNING_RATE), Trace_ELBO())
        svi_result = svi.run(rng_key, SVI_STEPS, arrays["X_train"], arrays["Xgp_train"], arrays["y_train"], spec["hsgp_m"], spec["hsgp_ell"], progress_bar=False)
        params = svi_result.params
        final_loss = float(svi_result.losses[-1])
        pred_key = jax.random.PRNGKey(SEED + fold_idx * 1000 + spec_idx)
        predictive = Predictive(spec["model"], guide=guide, params=params, num_samples=POSTERIOR_SAMPLES, return_sites=("mu", "obs", "hsgp_residual", "hsgp_alpha_value", "hsgp_length_value"))
        pred = predictive(pred_key, arrays["X_valid"], arrays["Xgp_valid"], None, spec["hsgp_m"], spec["hsgp_ell"])
        mu_draws_delta = np.asarray(pred["mu"]) * arrays["y_std"] + arrays["y_mean"]
        obs_draws_delta = np.asarray(pred["obs"]) * arrays["y_std"] + arrays["y_mean"]
        hsgp_draws_delta = np.asarray(pred["hsgp_residual"]) * arrays["y_std"]
        last_known = valid_frame["last_known_tvt"].to_numpy(float)[None, :]
        tvt_mu_draws = mu_draws_delta + last_known
        tvt_obs_draws = obs_draws_delta + last_known
        pred_tvt = tvt_mu_draws.mean(axis=0)
        metrics = metric_row(y_true_tvt, pred_tvt)
        intervals = interval_metrics(y_true_tvt, tvt_obs_draws)
        n_basis = int(np.prod(spec["hsgp_m"])) if len(spec["hsgp_m"]) else 0
        hsgp_alpha = float(np.asarray(pred.get("hsgp_alpha_value", np.array([np.nan]))).mean())
        hsgp_length = float(np.asarray(pred.get("hsgp_length_value", np.array([np.nan]))).mean())
        hsgp_residual_sd = float(hsgp_draws_delta.mean(axis=0).std())
        fold_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "guide": spec["guide"], "coord_set": spec["coord_set"], "n_basis": n_basis, "n_train_rows": len(train_frame), "n_valid_rows": len(valid_frame), "final_loss": final_loss, "hsgp_alpha": hsgp_alpha, "hsgp_length": hsgp_length, "hsgp_residual_sd": hsgp_residual_sd, **metrics, **intervals})
        posterior_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "final_loss": final_loss, "hsgp_alpha": hsgp_alpha, "hsgp_length": hsgp_length, "hsgp_residual_sd": hsgp_residual_sd, "n_basis": n_basis})
        sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), RESIDUAL_SAMPLE_PER_FOLD)).astype(int)
        residual_rows.append(pd.DataFrame({"fold": fold_idx, "candidate": spec["candidate"], "wellname": valid_frame.iloc[sample_idx]["wellname"].to_numpy(), "row_idx": valid_frame.iloc[sample_idx]["row_idx"].to_numpy(), "distance_from_boundary_md": valid_frame.iloc[sample_idx]["distance_from_boundary_md"].to_numpy(float), "eval_progress": valid_frame.iloc[sample_idx]["eval_progress"].to_numpy(float), "horizontal_dist_from_boundary": valid_frame.iloc[sample_idx]["horizontal_dist_from_boundary"].to_numpy(float), "y_true": y_true_tvt[sample_idx], "y_pred": pred_tvt[sample_idx], "residual": y_true_tvt[sample_idx] - pred_tvt[sample_idx], "hsgp_residual_mean": hsgp_draws_delta[:, sample_idx].mean(axis=0), "posterior_sd": tvt_obs_draws[:, sample_idx].std(axis=0)}))

fit_seconds = time.time() - fit_start
fold_metrics = pd.DataFrame(fold_rows)
summary = (fold_metrics.groupby(["candidate", "guide", "coord_set", "n_basis"], as_index=False).agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), coverage_80_mean=("coverage_80", "mean"), coverage_95_mean=("coverage_95", "mean"), width_80_mean=("width_80", "mean"), width_95_mean=("width_95", "mean"), hsgp_alpha_mean=("hsgp_alpha", "mean"), hsgp_length_mean=("hsgp_length", "mean"), hsgp_residual_sd_mean=("hsgp_residual_sd", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_train_rows_mean=("n_train_rows", "mean"), final_loss_mean=("final_loss", "mean")).sort_values("rmse_mean"))
residual_sample = pd.concat(residual_rows, ignore_index=True)
posterior_diagnostics = pd.DataFrame(posterior_rows)

print({"fit_seconds": round(fit_seconds, 2), "fold_rows": len(fold_metrics), "residual_rows": len(residual_sample)})
summary


{'fit_seconds': 138.04, 'fold_rows': 18, 'residual_rows': 27000}


,candidate,guide,coord_set,n_basis,rmse_mean,rmse_std,mae_mean,r2_mean,coverage_80_mean,coverage_95_mean,width_80_mean,width_95_mean,hsgp_alpha_mean,hsgp_length_mean,hsgp_residual_sd_mean,n_valid_rows,n_train_rows_mean,final_loss_mean
1,hsgp_bp_m5_autodelta,autodelta,boundary_progress,25,14.899696,0.510892,10.439902,0.999421,0.767462,0.931068,31.102881,58.658399,0.769609,0.802348,1.994644,108220,50000.0,64603.281250
4,hsgp_bph_m4_autodelta,autodelta,boundary_progress_horizontal,48,14.912554,0.507951,10.460338,0.999420,0.765154,0.930349,31.054326,58.589142,1.337030,0.723833,1.469942,108220,50000.0,64552.247396
5,linear_t3_autodelta_reference,autodelta,none,0,14.930657,0.579425,10.458673,0.999419,0.767352,0.931641,31.260024,58.971327,NaN,NaN,0.000000,108220,50000.0,64763.779948
3,hsgp_bp_m7_autodelta,autodelta,boundary_progress,49,14.930812,0.503155,10.473918,0.999418,0.765364,0.931539,31.098733,58.666239,0.706183,0.590996,2.118479,108220,50000.0,64619.981771
2,hsgp_bp_m5_autonormal,autonormal,boundary_progress,25,14.960790,0.485357,10.504409,0.999416,0.768660,0.931732,31.476230,59.287063,0.478349,0.500649,1.415077,108220,50000.0,65247.334635
0,endpoint_hold,baseline,baseline,0,15.578175,0.651555,10.905266,0.999367,NaN,NaN,NaN,NaN,NaN,NaN,NaN,108220,0.0,NaN


## 5. Artifacts And Diagnostics

In [6]:
"Export HSGP metrics, residual diagnostics, and figures."

fold_metrics_path = TABLE_DIR / f"model__bayesian_hsgp_residual_fold_metrics__{RUN_DATE}.csv"
summary_path = TABLE_DIR / f"model__bayesian_hsgp_residual_summary__{RUN_DATE}.csv"
posterior_path = TABLE_DIR / f"model__bayesian_hsgp_residual_posterior_diagnostics__{RUN_DATE}.csv"
residual_path = TABLE_DIR / f"model__bayesian_hsgp_residual_sample__{RUN_DATE}.csv"
fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
posterior_diagnostics.to_csv(posterior_path, index=False)
residual_sample.to_csv(residual_path, index=False)

fig_paths = {}
plot_summary = summary.sort_values("rmse_mean", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4C78A8" if candidate != "endpoint_hold" else "#F58518" for candidate in plot_summary["candidate"]]
ax.barh(plot_summary["candidate"], plot_summary["rmse_mean"], color=colors)
ax.set_title("Bayesian HSGP Residual RMSE")
ax.set_xlabel("grouped target-zone RMSE")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["rmse_comparison"] = FIGURE_DIR / f"model__bayesian_hsgp_residual_rmse_comparison__{RUN_DATE}.png"
fig.savefig(fig_paths["rmse_comparison"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
for candidate, group in fold_metrics.groupby("candidate"):
    if candidate == "endpoint_hold" or candidate in set(summary.head(4)["candidate"]):
        ax.plot(group["fold"], group["rmse"], marker="o", label=candidate)
ax.set_title("Bayesian HSGP Fold RMSE")
ax.set_xlabel("fold")
ax.set_ylabel("RMSE")
ax.legend(fontsize=7)
fig.tight_layout()
fig_paths["fold_rmse"] = FIGURE_DIR / f"model__bayesian_hsgp_residual_fold_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["fold_rmse"], dpi=160)
plt.close(fig)

best_candidate = str(summary.loc[summary["candidate"] != "endpoint_hold"].iloc[0]["candidate"])
best_residuals = residual_sample.loc[residual_sample["candidate"] == best_candidate]
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["distance_from_boundary_md"], best_residuals["hsgp_residual_mean"], s=7, alpha=0.35, color="#54A24B")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"HSGP Residual vs Boundary Distance: {best_candidate}")
ax.set_xlabel("MD distance past boundary")
ax.set_ylabel("HSGP residual mean")
fig.tight_layout()
fig_paths["hsgp_residual_by_boundary"] = FIGURE_DIR / f"model__bayesian_hsgp_residual_by_boundary__{RUN_DATE}.png"
fig.savefig(fig_paths["hsgp_residual_by_boundary"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["eval_progress"], best_residuals["residual"], s=7, alpha=0.35, color="#E45756")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"Residuals by Eval Progress: {best_candidate}")
ax.set_xlabel("eval progress")
ax.set_ylabel("TVT residual")
fig.tight_layout()
fig_paths["residual_by_progress"] = FIGURE_DIR / f"model__bayesian_hsgp_residual_by_progress__{RUN_DATE}.png"
fig.savefig(fig_paths["residual_by_progress"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
bayes_only = summary.loc[summary["candidate"] != "endpoint_hold"]
ax.scatter(bayes_only["width_80_mean"], bayes_only["coverage_80_mean"], s=80, color="#B279A2")
for row in bayes_only.itertuples(index=False):
    ax.text(row.width_80_mean, row.coverage_80_mean, row.candidate, fontsize=7)
ax.axhline(0.80, color="black", linestyle="--", linewidth=1)
ax.set_title("HSGP Coverage vs Width")
ax.set_xlabel("mean 80% interval width")
ax.set_ylabel("empirical 80% coverage")
fig.tight_layout()
fig_paths["coverage_width"] = FIGURE_DIR / f"model__bayesian_hsgp_residual_coverage_width__{RUN_DATE}.png"
fig.savefig(fig_paths["coverage_width"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}


{'rmse_comparison': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_hsgp_residual_rmse_comparison__2026-06-22.png',
 'fold_rmse': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_hsgp_residual_fold_rmse__2026-06-22.png',
 'hsgp_residual_by_boundary': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_hsgp_residual_by_boundary__2026-06-22.png',
 'residual_by_progress': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_hsgp_residual_by_progress__2026-06-22.png',
 'coverage_width': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_hsgp_residual_coverage_width__2026-06-22.png'}

In [7]:
"Write report, update comparison table, and create run bundle."

bayes_summary = summary.loc[summary["candidate"] != "endpoint_hold"].copy()
best = bayes_summary.iloc[0]
linear_reference = summary.loc[summary["candidate"] == "linear_t3_autodelta_reference"].iloc[0]
endpoint = summary.loc[summary["candidate"] == "endpoint_hold"].iloc[0]
previous_bayes_rmse = 14.844948333805148
huber_glm_rmse = 14.792842528391505
gam_public_rmse = 15.055

findings = pd.DataFrame([
    {"topic": "best_hsgp_candidate", "finding": f"Best HSGP candidate is {best.candidate} with sampled grouped RMSE {best.rmse_mean:.4f}.", "modeling_implication": "Use this as the first check of whether smooth GP residual structure improves Bayesian point predictions."},
    {"topic": "linear_reference_delta", "finding": f"Best HSGP RMSE delta versus same-notebook linear reference is {best.rmse_mean - linear_reference.rmse_mean:.4f}.", "modeling_implication": "A negative delta supports expanding HSGP; a small or positive delta means revisit coordinates, priors, or validation before making it more complex."},
    {"topic": "bayesian_baseline_delta", "finding": f"Best HSGP RMSE delta versus prior tuned Bayesian best is {best.rmse_mean - previous_bayes_rmse:.4f}.", "modeling_implication": "This controls for notebook-to-notebook stochastic and implementation differences."},
    {"topic": "huber_reference_delta", "finding": f"Best HSGP RMSE delta versus Huber GLM local reference is {best.rmse_mean - huber_glm_rmse:.4f}.", "modeling_implication": "If HSGP still trails Huber, use residual diagnostics to decide between richer HSGP and boosted trees."},
    {"topic": "posterior_calibration", "finding": f"Best HSGP 80% coverage is {best.coverage_80_mean:.3f} with mean width {best.width_80_mean:.2f}.", "modeling_implication": "Coverage and width indicate whether HSGP uncertainty remains usable for later hierarchical extensions."},
])
findings_path = TABLE_DIR / f"model__bayesian_hsgp_residual_findings__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)

report_path = REPORT_DIR / f"model__bayesian_hsgp_residual__{RUN_DATE}.md"
report_lines = ["# Bayesian HSGP Residual Baseline", "", f"Date: {RUN_DATE}", "", "## Key Findings"]
for row in findings.itertuples(index=False):
    report_lines.extend([f"- **{row.topic}**: {row.finding}", f"  - Modeling implication: {row.modeling_implication}"])
report_lines.extend(["", "## Best Candidate", f"- Candidate: `{best.candidate}`", f"- Guide: `{best.guide}`", f"- Coordinate set: `{best.coord_set}`", f"- Basis count: {int(best.n_basis)}", f"- RMSE mean/std: {best.rmse_mean:.4f} / {best.rmse_std:.4f}", f"- 80% coverage/width: {best.coverage_80_mean:.3f} / {best.width_80_mean:.2f}", f"- HSGP alpha/length: {best.hsgp_alpha_mean:.3f} / {best.hsgp_length_mean:.3f}", f"- Current best public LB remains GAM: {gam_public_rmse:.3f}", "", "## Artifacts", f"- Fold metrics: {fold_metrics_path}", f"- Summary: {summary_path}", f"- Posterior diagnostics: {posterior_path}", f"- Residual sample: {residual_path}", f"- Findings: {findings_path}"])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path)
run_comparison_row = pd.DataFrame([{"run_id": "pending_run_bundle", "model_family": "bayesian_hsgp_residual", "model_name": str(best.candidate), "cv_rmse_mean": float(best.rmse_mean), "cv_rmse_std": float(best.rmse_std), "train_time_s": float(build_seconds + fit_seconds), "n_features": int(len(feature_columns) + best.n_basis), "notes": "NumPyro/JAX HSGP residual baseline over boundary/progress coordinates"}])
comparison = comparison.loc[comparison["run_id"] != "pending_run_bundle"]
comparison = pd.concat([comparison, run_comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

config = {"task": "bayesian_hsgp_residual_baseline", "seed": SEED, "n_splits": N_SPLITS, "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL, "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD, "svi_steps": SVI_STEPS, "learning_rate": LEARNING_RATE, "posterior_samples": POSTERIOR_SAMPLES, "best_candidate": str(best.candidate), "note": "First HSGP residual baseline before richer GP and hierarchical models."}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_metric("sample_rows", float(len(feature_frame)))
    rb.add_metric("best_rmse", float(best.rmse_mean))
    rb.add_metric("best_rmse_std", float(best.rmse_std))
    rb.add_metric("best_coverage_80", float(best.coverage_80_mean))
    rb.add_metric("best_width_80", float(best.width_80_mean))
    rb.add_metric("linear_reference_rmse", float(linear_reference.rmse_mean))
    rb.add_metric("endpoint_hold_rmse", float(endpoint.rmse_mean))
    rb.add_metric("best_minus_linear_reference", float(best.rmse_mean - linear_reference.rmse_mean))
    rb.add_metric("best_minus_previous_bayes", float(best.rmse_mean - previous_bayes_rmse))
    rb.add_metric("best_minus_huber_glm", float(best.rmse_mean - huber_glm_rmse))
    for output_path in [fold_metrics_path, summary_path, posterior_path, residual_path, findings_path, report_path, comparison_path]:
        rb.add_output(str(output_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("First HSGP residual baseline completed with boundary/progress coordinate variants.")
run_id = rb.run_id
comparison.loc[comparison["run_id"] == "pending_run_bundle", "run_id"] = run_id
comparison.to_csv(comparison_path, index=False)

print({"run_id": run_id, "best_candidate": best.candidate, "best_rmse": round(float(best.rmse_mean), 4), "best_minus_linear_reference": round(float(best.rmse_mean - linear_reference.rmse_mean), 4), "best_minus_previous_bayes": round(float(best.rmse_mean - previous_bayes_rmse), 4), "coverage_80": round(float(best.coverage_80_mean), 3)})
findings


{'run_id': '2026-06-22__1358__bayesian_hsgp_residual_baseline__bd24319', 'best_candidate': 'hsgp_bp_m5_autodelta', 'best_rmse': 14.8997, 'best_minus_linear_reference': -0.031, 'best_minus_previous_bayes': 0.0547, 'coverage_80': 0.767}


,topic,finding,modeling_implication
0,best_hsgp_candidate,Best HSGP candidate is hsgp_bp_m5_autodelta wi...,Use this as the first check of whether smooth ...
1,linear_reference_delta,Best HSGP RMSE delta versus same-notebook line...,A negative delta supports expanding HSGP; a sm...
2,bayesian_baseline_delta,Best HSGP RMSE delta versus prior tuned Bayesi...,This controls for notebook-to-notebook stochas...
3,huber_reference_delta,Best HSGP RMSE delta versus Huber GLM local re...,"If HSGP still trails Huber, use residual diagn..."
4,posterior_calibration,Best HSGP 80% coverage is 0.767 with mean widt...,Coverage and width indicate whether HSGP uncer...


## Results

Run all cells to refresh the first HSGP residual baseline.
